In [ ]:
# test_prompt1.py
import pytest
from solution import Sheet

def test_basic_numbers():
    sheet = Sheet()
    sheet.set_cell("A1", 42)
    sheet.set_cell("B1", 3.14)
    assert sheet.get_value("A1") == 42
    assert sheet.get_value("B1") == 3.14

def test_basic_arithmetic():
    sheet = Sheet()
    sheet.set_cell("A1", "=2+3")
    sheet.set_cell("A2", "=10-4")
    sheet.set_cell("A3", "=5*6")
    sheet.set_cell("A4", "=20/4")
    sheet.set_cell("A5", "=2^3")
    assert sheet.get_value("A1") == 5
    assert sheet.get_value("A2") == 6
    assert sheet.get_value("A3") == 30
    assert sheet.get_value("A4") == 5
    assert sheet.get_value("A5") == 8

def test_parentheses():
    sheet = Sheet()
    sheet.set_cell("A1", "=(2+3)*4")
    sheet.set_cell("A2", "=2+3*4")
    assert sheet.get_value("A1") == 20
    assert sheet.get_value("A2") == 14

def test_cell_references():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("B1", "=A1+A2")
    assert sheet.get_value("B1") == 30

def test_chained_references():
    sheet = Sheet()
    sheet.set_cell("A1", 5)
    sheet.set_cell("A2", "=A1*2")
    sheet.set_cell("A3", "=A2+3")
    assert sheet.get_value("A3") == 13

def test_division_by_zero():
    sheet = Sheet()
    sheet.set_cell("A1", "=5/0")
    assert sheet.get_value("A1") == "#DIV/0!"
    
def test_division_by_zero_indirect():
    sheet = Sheet()
    sheet.set_cell("A1", 0)
    sheet.set_cell("A2", "=10/A1")
    assert sheet.get_value("A2") == "#DIV/0!"

def test_unknown_reference():
    sheet = Sheet()
    sheet.set_cell("A1", "=B2+5")
    # B2 is empty, should be treated as 0
    assert sheet.get_value("A1") == 5

def test_name_error_invalid_ref():
    sheet = Sheet()
    sheet.set_cell("A1", "=XYZ+5")
    assert sheet.get_value("A1") == "#NAME?"

def test_invalid_cell_reference():
    sheet = Sheet()
    sheet.set_cell("A1", "=A0")
    assert sheet.get_value("A1") == "#REF!"
    sheet.set_cell("A2", "=ZZZ999999")
    assert sheet.get_value("A2") == "#REF!"

def test_empty_cell_as_zero():
    sheet = Sheet()
    sheet.set_cell("A1", "=B1+5")
    assert sheet.get_value("A1") == 5

def test_value_error_type_mismatch():
    sheet = Sheet()
    sheet.set_cell("A1", "hello")
    sheet.set_cell("A2", "=A1+5")
    assert sheet.get_value("A2") == "#VALUE!"

def test_float_arithmetic():
    sheet = Sheet()
    sheet.set_cell("A1", "=3.5+2.5")
    assert sheet.get_value("A1") == 6.0

def test_negative_numbers():
    sheet = Sheet()
    sheet.set_cell("A1", "=-5+3")
    assert sheet.get_value("A1") == -2

def test_complex_formula():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("A3", "=(A1+A2)*2-5")
    assert sheet.get_value("A3") == 55

def test_multichar_columns():
    sheet = Sheet()
    sheet.set_cell("AA1", 100)
    sheet.set_cell("AB1", "=AA1+50")
    assert sheet.get_value("AB1") == 150
    
def test_valid_row_range():
    sheet = Sheet()
    sheet.set_cell("A1000", 42)
    assert sheet.get_value("A1000") == 42
    sheet.set_cell("A1", "=A1001")
    assert sheet.get_value("A1") == "#REF!"

In [ ]:
# test_prompt2.py
import pytest
from solution import Sheet

def test_simple_dependency_update():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("B1", "=A1*2")
    assert sheet.get_value("B1") == 20
    sheet.set_cell("A1", 15)
    assert sheet.get_value("B1") == 30

def test_chain_dependency_update():
    sheet = Sheet()
    sheet.set_cell("A1", 5)
    sheet.set_cell("B1", "=A1+1")
    sheet.set_cell("C1", "=B1+1")
    sheet.set_cell("D1", "=C1+1")
    assert sheet.get_value("D1") == 8
    sheet.set_cell("A1", 10)
    assert sheet.get_value("D1") == 13

def test_simple_cycle():
    sheet = Sheet()
    sheet.set_cell("A1", "=B1")
    sheet.set_cell("B1", "=A1")
    assert sheet.get_value("A1") == "#CYCLE!"
    assert sheet.get_value("B1") == "#CYCLE!"

def test_three_cell_cycle():
    sheet = Sheet()
    sheet.set_cell("A1", "=B1")
    sheet.set_cell("B1", "=C1")
    sheet.set_cell("C1", "=A1")
    assert sheet.get_value("A1") == "#CYCLE!"
    assert sheet.get_value("B1") == "#CYCLE!"
    assert sheet.get_value("C1") == "#CYCLE!"

def test_self_reference_cycle():
    sheet = Sheet()
    sheet.set_cell("A1", "=A1+1")
    assert sheet.get_value("A1") == "#CYCLE!"

def test_dependent_on_cycle():
    sheet = Sheet()
    sheet.set_cell("A1", "=B1")
    sheet.set_cell("B1", "=A1")
    sheet.set_cell("C1", "=A1+10")
    assert sheet.get_value("C1") == "#CYCLE!"

def test_breaking_cycle():
    sheet = Sheet()
    sheet.set_cell("A1", "=B1")
    sheet.set_cell("B1", "=A1")
    assert sheet.get_value("A1") == "#CYCLE!"
    sheet.set_cell("B1", 10)
    assert sheet.get_value("A1") == 10
    assert sheet.get_value("B1") == 10

def test_partial_cycle():
    sheet = Sheet()
    sheet.set_cell("A1", 5)
    sheet.set_cell("B1", "=A1")
    sheet.set_cell("C1", "=D1")
    sheet.set_cell("D1", "=C1")
    assert sheet.get_value("B1") == 5
    assert sheet.get_value("C1") == "#CYCLE!"
    assert sheet.get_value("D1") == "#CYCLE!"

def test_diamond_dependency():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("B1", "=A1+5")
    sheet.set_cell("C1", "=A1+10")
    sheet.set_cell("D1", "=B1+C1")
    assert sheet.get_value("D1") == 35
    sheet.set_cell("A1", 20)
    assert sheet.get_value("D1") == 55

def test_no_unnecessary_recalc():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("B1", 20)
    sheet.set_cell("C1", "=A1")
    sheet.set_cell("D1", "=B1")
    # Updating A1 should not affect D1
    v1 = sheet.get_value("D1")
    sheet.set_cell("A1", 999)
    assert sheet.get_value("D1") == v1

In [ ]:
import pytest
from solution import Sheet

def test_sum_simple_range():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("A3", 30)
    sheet.set_cell("B1", "=SUM(A1:A3)")
    assert sheet.get_value("B1") == 60

def test_sum_with_blanks():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A3", 30)
    sheet.set_cell("B1", "=SUM(A1:A3)")
    assert sheet.get_value("B1") == 40

def test_average_simple():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("A3", 30)
    sheet.set_cell("B1", "=AVERAGE(A1:A3)")
    assert sheet.get_value("B1") == 20

def test_average_with_blanks():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("B1", "=AVERAGE(A1:A3)")
    # A3 is blank, counts as 0
    assert sheet.get_value("B1") == 10

def test_average_all_blanks():
    sheet = Sheet()
    sheet.set_cell("B1", "=AVERAGE(A1:A3)")
    assert sheet.get_value("B1") == "#DIV/0!"

def test_min_simple():
    sheet = Sheet()
    sheet.set_cell("A1", 30)
    sheet.set_cell("A2", 10)
    sheet.set_cell("A3", 20)
    sheet.set_cell("B1", "=MIN(A1:A3)")
    assert sheet.get_value("B1") == 10

def test_min_with_blanks():
    sheet = Sheet()
    sheet.set_cell("A1", 30)
    sheet.set_cell("A3", 20)
    sheet.set_cell("B1", "=MIN(A1:A3)")
    assert sheet.get_value("B1") == 0

def test_min_all_blanks():
    sheet = Sheet()
    sheet.set_cell("B1", "=MIN(A1:A3)")
    assert sheet.get_value("B1") == "#N/A!"

def test_max_simple():
    sheet = Sheet()
    sheet.set_cell("A1", 30)
    sheet.set_cell("A2", 10)
    sheet.set_cell("A3", 20)
    sheet.set_cell("B1", "=MAX(A1:A3)")
    assert sheet.get_value("B1") == 30

def test_max_all_blanks():
    sheet = Sheet()
    sheet.set_cell("B1", "=MAX(A1:A3)")
    assert sheet.get_value("B1") == "#N/A!"

def test_rectangular_range():
    sheet = Sheet()
    sheet.set_cell("A1", 1)
    sheet.set_cell("A2", 2)
    sheet.set_cell("B1", 3)
    sheet.set_cell("B2", 4)
    sheet.set_cell("C1", "=SUM(A1:B2)")
    assert sheet.get_value("C1") == 10

def test_error_propagation():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", "=5/0")
    sheet.set_cell("A3", 20)
    sheet.set_cell("B1", "=SUM(A1:A3)")
    assert sheet.get_value("B1") == "#DIV/0!"

def test_error_propagation_order():
    sheet = Sheet()
    sheet.set_cell("A1", "=5/0")
    sheet.set_cell("A2", "=1/0")
    sheet.set_cell("B1", "=SUM(A1:A2)")
    # Should propagate first error (A1)
    assert sheet.get_value("B1") == "#DIV/0!"

def test_single_cell_range():
    sheet = Sheet()
    sheet.set_cell("A1", 42)
    sheet.set_cell("B1", "=SUM(A1:A1)")
    assert sheet.get_value("B1") == 42

def test_range_with_formulas():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", "=A1*2")
    sheet.set_cell("A3", "=A2+5")
    sheet.set_cell("B1", "=SUM(A1:A3)")
    assert sheet.get_value("B1") == 55

def test_nested_ranges_not_supported():
    sheet = Sheet()
    sheet.set_cell("A1", "=SUM(A1:A2, B1:B2)")
    assert sheet.get_value("A1") == "#VALUE!"

In [ ]:
import pytest
from solution import Sheet

def test_absolute_column_and_row():
    sheet = Sheet()
    sheet.set_cell("A1", "=$B$2")
    sheet.set_cell("B2", 100)
    sheet.copy_paste("A1", "C3")
    assert sheet.get_value("C3") == 100

def test_absolute_column_relative_row():
    sheet = Sheet()
    sheet.set_cell("A1", "=$B1")
    sheet.set_cell("B1", 10)
    sheet.set_cell("B2", 20)
    sheet.copy_paste("A1", "A2")
    # $B1 -> $B2 (row shifts)
    assert sheet.get_value("A2") == 20

def test_relative_column_absolute_row():
    sheet = Sheet()
    sheet.set_cell("A1", "=A$2")
    sheet.set_cell("A2", 10)
    sheet.set_cell("B2", 20)
    sheet.copy_paste("A1", "B1")
    # A$2 -> B$2 (column shifts)
    assert sheet.get_value("B1") == 20

def test_fully_relative():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("B1", "=A1")
    sheet.copy_paste("B1", "C2")
    # A1 -> B2 (both shift by +1 col, +1 row)
    sheet.set_cell("B2", 20)
    assert sheet.get_value("C2") == 20

def test_copy_value():
    sheet = Sheet()
    sheet.set_cell("A1", 42)
    sheet.copy_paste("A1", "B2")
    assert sheet.get_value("B2") == 42

def test_copy_empty_clears():
    sheet = Sheet()
    sheet.set_cell("A1", 100)
    sheet.copy_paste("B1", "A1")  # B1 empty -> clears A1
    assert sheet.get_value("A1") == 0

def test_paste_does_not_change_source():
    sheet = Sheet()
    sheet.set_cell("A1", "=B1+5")
    sheet.set_cell("B1", 10)
    sheet.copy_paste("A1", "A2")
    assert sheet.get_value("A1") == 15

def test_reference_out_of_bounds_column():
    sheet = Sheet()
    # Columns are capped at ZZZ. Copying right should turn ZZZ1 -> AAAA1, which is out of bounds -> #REF!
    sheet.set_cell("A1", "=ZZZ1")
    sheet.copy_paste("A1", "B1")
    assert sheet.get_value("B1") == "#REF!"

def test_reference_out_of_bounds_row():
    sheet = Sheet()
    # Rows are capped at 1000. Shifting A1000 down by 1 row -> A1001, which is out of bounds -> #REF!
    sheet.set_cell("A998", "=A1000")
    sheet.copy_paste("A998", "A999")
    assert sheet.get_value("A999") == "#REF!"

def test_complex_formula_copy():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("B1", "=$A1+A2")
    sheet.copy_paste("B1", "C2")
    # $A1 stays A1, A2 -> B3
    sheet.set_cell("B3", 30)
    assert sheet.get_value("C2") == 40

def test_multiple_references_in_formula():
    sheet = Sheet()
    sheet.set_cell("A1", "=B1+$C1+D$2")
    sheet.set_cell("B1", 1)
    sheet.set_cell("C1", 2)
    sheet.set_cell("D2", 3)
    sheet.copy_paste("A1", "A2")
    # B1 -> B2 (relative), $C1 -> $C2 (absolute column, relative row), D$2 -> D$2 (relative col, absolute row)
    sheet.set_cell("B2", 10)
    sheet.set_cell("C2", 20)
    assert sheet.get_value("A2") == 33

In [ ]:
# test_prompt5.py
import pytest
from solution import Sheet

def test_string_literal():
    sheet = Sheet()
    sheet.set_cell("A1", "=\"hello\"")
    assert sheet.get_value("A1") == "hello"

def test_concatenation():
    sheet = Sheet()
    sheet.set_cell("A1", "=\"hello\"&\" \"&\"world\"")
    assert sheet.get_value("A1") == "hello world"

def test_concatenation_with_cell():
    sheet = Sheet()
    sheet.set_cell("A1", "hello")
    sheet.set_cell("A2", "=A1&\" world\"")
    assert sheet.get_value("A2") == "hello world"

def test_number_string_concat_error():
    sheet = Sheet()
    sheet.set_cell("A1", "=5&\"text\"")
    assert sheet.get_value("A1") == "#VALUE!"

def test_empty_cell_as_empty_string():
    sheet = Sheet()
    sheet.set_cell("A1", "=\"hello\"&B1&\"world\"")
    assert sheet.get_value("A1") == "helloworld"

def test_text_function_integer():
    sheet = Sheet()
    sheet.set_cell("A1", "=TEXT(42, \"0\")")
    assert sheet.get_value("A1") == "42"

def test_text_function_float():
    sheet = Sheet()
    sheet.set_cell("A1", "=TEXT(3.14159, \"0.00\")")
    assert sheet.get_value("A1") == "3.14"

def test_value_function():
    sheet = Sheet()
    sheet.set_cell("A1", "=\"123\"")
    sheet.set_cell("A2", "=VALUE(A1)")
    assert sheet.get_value("A2") == 123

def test_value_function_error():
    sheet = Sheet()
    sheet.set_cell("A1", "=VALUE(\"abc\")")
    assert sheet.get_value("A1") == "#VALUE!"

def test_concatenate_function():
    sheet = Sheet()
    sheet.set_cell("A1", "=CONCATENATE(\"hello\", \" \", \"world\")")
    assert sheet.get_value("A1") == "hello world"

def test_concatenate_auto_convert_number():
    sheet = Sheet()
    sheet.set_cell("A1", "=CONCATENATE(\"Value: \", 42)")
    assert sheet.get_value("A1") == "Value: 42"

def test_string_in_arithmetic_error():
    sheet = Sheet()
    sheet.set_cell("A1", "=\"5\"")
    sheet.set_cell("A2", "=A1+10")
    assert sheet.get_value("A2") == "#VALUE!"

def test_text_preserves_values():
    sheet = Sheet()
    sheet.set_cell("A1", 100)
    sheet.set_cell("A2", "=TEXT(A1, \"0\")")
    sheet.set_cell("A3", "=A2&\" units\"")
    assert sheet.get_value("A3") == "100 units"

def test_concatenate_multiple_args():
    sheet = Sheet()
    sheet.set_cell("A1", "a")
    sheet.set_cell("A2", "b")
    sheet.set_cell("A3", "c")
    sheet.set_cell("B1", "=CONCATENATE(A1, A2, A3)")
    assert sheet.get_value("B1") == "abc"

def test_mixed_operations():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", "=TEXT(A1*2, \"0\")&\" items\"")
    assert sheet.get_value("A2") == "20 items"

In [ ]:
import pytest
from solution import Sheet
from datetime import datetime

def test_date_function():
    sheet = Sheet()
    sheet.set_cell("A1", "=DATE(2025, 1, 1)")
    # 1900 date system with Excel leap-year bug: 2025-01-01 is 45658
    assert sheet.get_value("A1") == 45658

def test_date_arithmetic():
    sheet = Sheet()
    sheet.set_cell("A1", "=DATE(2025, 1, 10) - DATE(2025, 1, 5)")
    assert sheet.get_value("A1") == 5

def test_today_with_clock():
    sheet = Sheet()
    sheet.set_clock(datetime(2025, 1, 1, 0, 0, 0))
    sheet.set_cell("A1", "=TODAY()")
    assert sheet.get_value("A1") == 45658

def test_now_with_clock():
    sheet = Sheet()
    sheet.set_clock(datetime(2025, 1, 1, 12, 0, 0))
    sheet.set_cell("A1", "=NOW()")
    # Should be 45658 + 0.5 (12 hours)
    assert abs(sheet.get_value("A1") - 45658.5) < 0.001

def test_time_function():
    sheet = Sheet()
    sheet.set_cell("A1", "=TIME(12, 0, 0)")
    assert abs(sheet.get_value("A1") - 0.5) < 0.001

def test_time_function_full_day():
    sheet = Sheet()
    sheet.set_cell("A1", "=TIME(6, 30, 0)")
    # 6.5 hours / 24 hours
    expected = 6.5 / 24
    assert abs(sheet.get_value("A1") - expected) < 0.001

def test_invalid_date():
    sheet = Sheet()
    sheet.set_cell("A1", "=DATE(2025, 13, 1)")
    assert sheet.get_value("A1") == "#VALUE!"

def test_invalid_date_day():
    sheet = Sheet()
    sheet.set_cell("A1", "=DATE(2025, 2, 30)")
    assert sheet.get_value("A1") == "#VALUE!"

def test_date_addition():
    sheet = Sheet()
    sheet.set_cell("A1", "=DATE(2025, 1, 1) + 10")
    sheet.set_cell("A2", "=DATE(2025, 1, 11)")
    assert sheet.get_value("A1") == sheet.get_value("A2")

def test_date_in_formula():
    sheet = Sheet()
    sheet.set_cell("A1", "=DATE(2025, 1, 15)")
    sheet.set_cell("A2", "=A1 - 5")
    sheet.set_cell("A3", "=DATE(2025, 1, 10)")
    assert sheet.get_value("A2") == sheet.get_value("A3")

def test_default_clock():
    sheet = Sheet()
    # Should have a default fixed time
    sheet.set_cell("A1", "=TODAY()")
    v1 = sheet.get_value("A1")
    assert isinstance(v1, (int, float))

def test_time_with_date():
    sheet = Sheet()
    sheet.set_cell("A1", "=DATE(2025, 1, 1) + TIME(12, 30, 0)")
    # Should be date serial + time fraction (using 45658)
    result = sheet.get_value("A1")
    assert 45658 < result < 45659

In [ ]:
# test_prompt7.py
import pytest
from solution import Sheet

def test_rand_function():
    sheet = Sheet()
    sheet.set_seed(42)
    sheet.set_cell("A1", "=RAND()")
    v1 = sheet.get_value("A1")
    assert 0 <= v1 < 1

def test_rand_deterministic():
    sheet1 = Sheet()
    sheet1.set_seed(42)
    sheet1.set_cell("A1", "=RAND()")
    v1 = sheet1.get_value("A1")
    
    sheet2 = Sheet()
    sheet2.set_seed(42)
    sheet2.set_cell("A1", "=RAND()")
    v2 = sheet2.get_value("A1")
    
    assert v1 == v2

def test_randbetween():
    sheet = Sheet()
    sheet.set_seed(42)
    sheet.set_cell("A1", "=RANDBETWEEN(1, 10)")
    v1 = sheet.get_value("A1")
    assert 1 <= v1 <= 10
    assert isinstance(v1, int)

def test_volatile_recalculation():
    sheet = Sheet()
    sheet.set_seed(42)
    sheet.set_cell("A1", "=RAND()")
    v1 = sheet.get_value("A1")
    sheet.recalculate()
    v2 = sheet.get_value("A1")
    # Should be different after recalculation
    assert v1 != v2

def test_non_volatile_no_recalc():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("B1", "=A1*2")
    v1 = sheet.get_value("B1")
    sheet.recalculate()
    v2 = sheet.get_value("B1")
    # Should be same since A1 didn't change
    assert v1 == v2

def test_volatile_propagates():
    sheet = Sheet()
    sheet.set_seed(42)
    sheet.set_cell("A1", "=RAND()")
    sheet.set_cell("B1", "=A1*2")
    v1 = sheet.get_value("B1")
    sheet.recalculate()
    v2 = sheet.get_value("B1")
    # B1 should also recalculate
    assert v1 != v2

def test_nested_volatile():
    sheet = Sheet()
    sheet.set_seed(42)
    sheet.set_cell("A1", "=RAND()+5")
    v1 = sheet.get_value("A1")
    sheet.recalculate()
    v2 = sheet.get_value("A1")
    assert v1 != v2

def test_randbetween_range():
    sheet = Sheet()
    sheet.set_seed(42)
    results = []
    for i in range(100):
        sheet.set_cell("A1", "=RANDBETWEEN(5, 5)")
        sheet.recalculate()
        results.append(sheet.get_value("A1"))
    assert all(r == 5 for r in results)

def test_multiple_volatile_cells():
    sheet = Sheet()
    sheet.set_seed(42)
    sheet.set_cell("A1", "=RAND()")
    sheet.set_cell("A2", "=RAND()")
    v1_1 = sheet.get_value("A1")
    v2_1 = sheet.get_value("A2")
    sheet.recalculate()
    v1_2 = sheet.get_value("A1")
    v2_2 = sheet.get_value("A2")
    assert v1_1 != v1_2
    assert v2_1 != v2_2

def test_volatile_with_dependency():
    sheet = Sheet()
    sheet.set_seed(42)
    sheet.set_cell("A1", 10)
    sheet.set_cell("B1", "=RAND()*A1")
    v1 = sheet.get_value("B1")
    # Change non-volatile dependency
    sheet.set_cell("A1", 20)
    v2 = sheet.get_value("B1")
    # Should recalculate
    assert v1 != v2

In [ ]:
import pytest
from solution import Sheet

def test_simple_array_spill():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("A3", 30)
    sheet.set_array_formula("B1", "=A1:A3*2")
    assert sheet.get_value("B1") == 20
    assert sheet.get_value("B2") == 40
    assert sheet.get_value("B3") == 60

def test_spill_collision():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("B2", 999)  # Blocks spill
    sheet.set_array_formula("B1", "=A1:A2*2")
    assert sheet.get_value("B1") == "#SPILL!"
    assert sheet.get_value("B2") == 999

def test_clear_array_formula():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_array_formula("B1", "=A1:A2*2")
    assert sheet.get_value("B1") == 20
    assert sheet.get_value("B2") == 40
    sheet.clear_cell("B1")
    # Spill should be cleared
    assert sheet.get_value("B2") == 0

def test_mismatched_shapes():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("A3", 30)
    sheet.set_cell("C1", 5)
    sheet.set_cell("C2", 6)
    sheet.set_array_formula("B1", "=A1:A3+C1:C2")
    assert sheet.get_value("B1") == "#N/A!"

def test_scalar_broadcast():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_array_formula("B1", "=A1:A2+5")
    assert sheet.get_value("B1") == 15
    assert sheet.get_value("B2") == 25

def test_rectangular_array():
    sheet = Sheet()
    sheet.set_cell("A1", 1)
    sheet.set_cell("A2", 2)
    sheet.set_cell("B1", 3)
    sheet.set_cell("B2", 4)
    sheet.set_array_formula("C1", "=A1:B2*2")
    assert sheet.get_value("C1") == 2
    assert sheet.get_value("C2") == 4
    assert sheet.get_value("D1") == 6
    assert sheet.get_value("D2") == 8

def test_spill_readonly():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_array_formula("B1", "=A1:A2*2")
    # Try to set spill cell directly
    sheet.set_cell("B2", 999)
    # Spill cell should remain computed (read-only)
    assert sheet.get_value("B2") == 40

def test_array_with_formulas():
    sheet = Sheet()
    sheet.set_cell("A1", 5)
    sheet.set_cell("A2", "=A1*2")
    sheet.set_array_formula("B1", "=A1:A2+10")
    assert sheet.get_value("B1") == 15
    assert sheet.get_value("B2") == 20

def test_single_cell_array():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_array_formula("B1", "=A1:A1*3")
    assert sheet.get_value("B1") == 30

def test_array_dependency_update():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_array_formula("B1", "=A1:A2*2")
    assert sheet.get_value("B2") == 40
    sheet.set_cell("A2", 30)
    assert sheet.get_value("B2") == 60

In [ ]:
# test_prompt9.py
import pytest
from solution import Sheet

def test_index_simple():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("A3", 30)
    sheet.set_cell("B1", "=INDEX(A1:A3, 2, 1)")
    assert sheet.get_value("B1") == 20

def test_index_rectangular():
    sheet = Sheet()
    sheet.set_cell("A1", 1)
    sheet.set_cell("A2", 2)
    sheet.set_cell("B1", 3)
    sheet.set_cell("B2", 4)
    sheet.set_cell("C1", "=INDEX(A1:B2, 2, 2)")
    assert sheet.get_value("C1") == 4

def test_index_out_of_bounds():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("B1", "=INDEX(A1:A3, 5, 1)")
    assert sheet.get_value("B1") == "#REF!"

def test_match_exact():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("A3", 30)
    sheet.set_cell("B1", "=MATCH(20, A1:A3, 0)")
    assert sheet.get_value("B1") == 2

def test_match_not_found():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("B1", "=MATCH(25, A1:A2, 0)")
    assert sheet.get_value("B1") == "#N/A!"

def test_match_approximate_ascending():
    sheet = Sheet()
    sheet.set_cell("A1", 10)
    sheet.set_cell("A2", 20)
    sheet.set_cell("A3", 30)
    sheet.set_cell("B1", "=MATCH(25, A1:A3, 1)")
    # Should return 2 (largest <= 25 is 20)
    assert sheet.get_value("B1") == 2

def test_match_approximate_descending():
    sheet = Sheet()
    sheet.set_cell("A1", 30)
    sheet.set_cell("A2", 20)
    sheet.set_cell("A3", 10)
    sheet.set_cell("B1", "=MATCH(15, A1:A3, -1)")
    # Should return 2 (smallest >= 15 is 20)
    assert sheet.get_value("B1") == 2

def test_vlookup_exact():
    sheet = Sheet()
    sheet.set_cell("A1", 100)
    sheet.set_cell("B1", "Apple")
    sheet.set_cell("A2", 200)
    sheet.set_cell("B2", "Banana")
    sheet.set_cell("C1", "=VLOOKUP(200, A1:B2, 2, 0)")
    assert sheet.get_value("C1") == "Banana"

def test_vlookup_not_found():
    sheet = Sheet()
    sheet.set_cell("A1", 100)
    sheet.set_cell("B1", "Apple")
    sheet.set_cell("C1", "=VLOOKUP(300, A1:B1, 2, 0)")
    assert sheet.get_value("C1") == "#N/A!"

def test_vlookup_approximate():
    sheet = Sheet()
    sheet.set_cell("A1", 0)
    sheet.set_cell("B1", "F")
    sheet.set_cell("A2", 60)
    sheet.set_cell("B2", "D")
    sheet.set_cell("A3", 70)
    sheet.set_cell("B3", "C")
    sheet.set_cell("A4", 80)
    sheet.set_cell("B4", "B")
    sheet.set_cell("C1", "=VLOOKUP(75, A1:B4, 2, 1)")
    # Should return "C" (largest <= 75 is 70)
    assert sheet.get_value("C1") == "C"

def test_vlookup_col_out_of_bounds():
    sheet = Sheet()
    sheet.set_cell("A1", 100)
    sheet.set_cell("B1", "Apple")
    sheet.set_cell("C1", "=VLOOKUP(100, A1:B1, 3, 0)")
    assert sheet.get_value("C1") == "#REF!"

def test_index_match_combo():
    sheet = Sheet()
    sheet.set_cell("A1", "Apple")
    sheet.set_cell("A2", "Banana")
    sheet.set_cell("A3", "Cherry")
    sheet.set_cell("B1", 10)
    sheet.set_cell("B2", 20)
    sheet.set_cell("B3", 30)
    sheet.set_cell("C1", "=INDEX(B1:B3, MATCH(\"Banana\", A1:A3, 0), 1)")
    assert sheet.get_value("C1") == 20

def test_vlookup_first_match_on_tie():
    sheet = Sheet()
    sheet.set_cell("A1", 100)
    sheet.set_cell("B1", "First")
    sheet.set_cell("A2", 100)
    sheet.set_cell("B2", "Second")
    sheet.set_cell("C1", "=VLOOKUP(100, A1:B2, 2, 0)")
    assert sheet.get_value("C1") == "First"

In [ ]:
import pytest
from solution import Sheet

def test_simple_iteration_convergence():
    sheet = Sheet()
    sheet.enable_iterations(100, 0.001)
    sheet.set_cell("A1", "=0.5*B1+1")
    sheet.set_cell("B1", "=0.5*A1")
    # Fixed point: A1 = 4/3, B1 = 2/3
    assert abs(sheet.get_value("A1") - (4/3)) < 1e-2
    assert abs(sheet.get_value("B1") - (2/3)) < 1e-2

def test_iteration_not_converged():
    sheet = Sheet()
    sheet.enable_iterations(5, 0.001)
    sheet.set_cell("A1", "=A1+1")
    # Can't converge
    assert sheet.get_value("A1") == "#CYCLE!"

def test_iteration_disabled_by_default():
    sheet = Sheet()
    sheet.set_cell("A1", "=B1")
    sheet.set_cell("B1", "=A1")
    assert sheet.get_value("A1") == "#CYCLE!"

def test_iteration_count_tracking():
    sheet = Sheet()
    sheet.enable_iterations(100, 0.001)
    sheet.set_cell("A1", "=0.5*B1+1")
    sheet.set_cell("B1", "=0.5*A1")
    sheet.get_value("A1")
    count = sheet.get_iteration_count()
    assert count > 0

def test_fast_convergence():
    sheet = Sheet()
    sheet.enable_iterations(100, 0.001)
    sheet.set_cell("A1", "=B1")
    sheet.set_cell("B1", 10)
    sheet.set_cell("B1", "=A1")
    sheet.set_cell("A1", 10)
    # Should converge immediately
    assert sheet.get_value("A1") == 10
    assert sheet.get_value("B1") == 10

def test_tolerance_check():
    sheet = Sheet()
    sheet.enable_iterations(100, 1.0)
    sheet.set_cell("A1", "=B1+0.5")
    sheet.set_cell("B1", "=A1-0.4")
    # With high tolerance, engine may stop early; just assert numeric outputs
    v1 = sheet.get_value("A1")
    v2 = sheet.get_value("B1")
    assert isinstance(v1, (int, float))
    assert isinstance(v2, (int, float))

def test_multiple_cycles_separate():
    sheet = Sheet()
    sheet.enable_iterations(100, 0.001)
    sheet.set_cell("A1", "=0.5*B1+1")
    sheet.set_cell("B1", "=0.5*A1")
    sheet.set_cell("C1", "=D1+2")
    sheet.set_cell("D1", "=C1-1")
    a1 = sheet.get_value("A1")
    c1 = sheet.get_value("C1")
    assert isinstance(a1, (int, float))
    assert isinstance(c1, (int, float))

def test_non_circular_with_iteration_on():
    sheet = Sheet()
    sheet.enable_iterations(100, 0.001)
    sheet.set_cell("A1", 10)
    sheet.set_cell("B1", "=A1*2")
    assert sheet.get_value("B1") == 20

def test_string_in_iteration():
    sheet = Sheet()
    sheet.enable_iterations(100, 0.001)
    sheet.set_cell("A1", "=B1")
    sheet.set_cell("B1", "hello")
    sheet.set_cell("B1", "=A1")
    # Cyclic non-numeric dependency should not converge -> #CYCLE!
    assert sheet.get_value("A1") == "#CYCLE!"

def test_complex_cycle_convergence():
    sheet = Sheet()
    sheet.enable_iterations(100, 0.001)
    sheet.set_cell("A1", "=B1+C1")
    sheet.set_cell("B1", "=A1/2")
    sheet.set_cell("C1", "=A1/2")
    result = sheet.get_value("A1")
    assert isinstance(result, (int, float))